# Scenario 2 — Databricks UC Federation + Snowpark Connect Demo

**What this demonstrates:**
- Snowflake federates Databricks Unity Catalog Iceberg tables via a Catalog Integration
- Snowpark Connect queries federated tables with Snowflake Horizon governance enforced
- Masking policies on `credit_card` apply across all Spark DataFrame operations

**Step 0–4** run once to set up (idempotent — safe to re-run).  
**Demos 1–9** can be re-run independently without repeating setup.

**Prerequisites:** Run `04_databricks_create_uc_tables.py` in Databricks first.

In [ ]:
# ----------------------------------------------------------------
# Parameters — SET YOUR VALUES HERE (change only this block)
# ----------------------------------------------------------------
DBX_WORKSPACE_HOST = "<DBX_WORKSPACE_HOST>"   # !! REPLACE: e.g. 'adb-1234567890.12.azuredatabricks.net'
DBX_UC_CATALOG     = "<DBX_UC_CATALOG>"       # !! REPLACE: Unity Catalog name (e.g. 'my_demo')
DBX_PAT_TOKEN      = "<DBX_PAT_TOKEN>"        # !! REPLACE: Databricks PAT — never commit
SF_WAREHOUSE       = "LOAD_WH"                # !! REPLACE if your warehouse differs
SF_USERNAME        = "<YOUR_USERNAME>"         # !! REPLACE: your Snowflake username

# Pre-set demo values — no changes needed below this line
SF_INIT_DB         = "HORIZON_DEMO_SFDB"
SF_FEDERATED_DB    = "DATABRICKS_DEMO_DB"
SF_FEDERATED_SCHEMA = "horizon_demo"
SF_EXTERNAL_VOLUME = "ICEBERG_EXTERNAL_S3_VOLUME"
SF_GOVERNANCE_DB   = "HORIZON_DEMO_SFDB"
SF_READER_ROLE     = "EXT_COMPUTE_ENG_DEMO_ROLE"

TBL_ORDERS    = f"{SF_FEDERATED_DB}.{SF_FEDERATED_SCHEMA}.customer_orders"
TBL_SENSITIVE = f"{SF_FEDERATED_DB}.{SF_FEDERATED_SCHEMA}.sensitive_orders"

print(f"Federated DB : {SF_FEDERATED_DB}.{SF_FEDERATED_SCHEMA}")
print(f"Warehouse    : {SF_WAREHOUSE}")

## Step 0 — Initialize Snowpark Connect Session

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake import snowpark_connect
from pyspark import SparkConf
from pyspark.sql.functions import col, count, sum as spark_sum
import pandas as pd

sf_session = get_active_session()
sf_session.sql(f"USE DATABASE {SF_INIT_DB}").collect()
sf_session.sql(f"USE WAREHOUSE {SF_WAREHOUSE}").collect()

conf = SparkConf().set("spark.sql.caseSensitive", "true")
spark = snowpark_connect.init_spark_session(conf=conf)

def switch_role(role: str):
    sf_session.sql(f"USE ROLE {role}").collect()
    print(f"Active role → {role}")

switch_role("ACCOUNTADMIN")
print("Sessions ready")

## Step 1 — Create Catalog Integration

Connects Snowflake to Databricks Unity Catalog's Iceberg REST endpoint.  
`CREATE OR REPLACE` is safe to re-run — updates the PAT if it has been rotated.

In [ ]:
switch_role("ACCOUNTADMIN")
sf_session.sql(f"""
    CREATE OR REPLACE CATALOG INTEGRATION MY_DATABRICKS_UC_CI
        CATALOG_SOURCE = ICEBERG_REST
        TABLE_FORMAT   = ICEBERG
        REST_CONFIG = (
            CATALOG_URI            = 'https://{DBX_WORKSPACE_HOST}/api/2.1/unity-catalog/iceberg-rest'
            CATALOG_NAME           = '{DBX_UC_CATALOG}'
            ACCESS_DELEGATION_MODE = EXTERNAL_VOLUME_CREDENTIALS
        )
        REST_AUTHENTICATION = (
            TYPE         = BEARER
            BEARER_TOKEN = '{DBX_PAT_TOKEN}'
        )
        ENABLED = TRUE
""").collect()
print("Catalog Integration created ✅")

verify = sf_session.sql("SELECT SYSTEM$VERIFY_CATALOG_INTEGRATION('MY_DATABRICKS_UC_CI')").collect()[0][0]
print(f"Connectivity: {verify}")

## Step 2 — Create Catalog-Linked Database

`CREATE DATABASE IF NOT EXISTS` — completely safe to re-run.  
Auto-discovers schemas and tables from the Databricks UC catalog every 30 seconds.

In [ ]:
sf_session.sql(f"""
    CREATE DATABASE IF NOT EXISTS {SF_FEDERATED_DB}
        EXTERNAL_VOLUME = '{SF_EXTERNAL_VOLUME}'
        LINKED_CATALOG = ( CATALOG = 'MY_DATABRICKS_UC_CI' )
        COMMENT = 'Iceberg Federation Demo — Federated from Databricks Unity Catalog'
""").collect()
print(f"Catalog-Linked Database '{SF_FEDERATED_DB}' ready ✅")

# List discovered tables
tables = sf_session.sql(f"SHOW ICEBERG TABLES IN DATABASE {SF_FEDERATED_DB}").to_pandas()
print(f"Discovered tables: {len(tables)}")
if len(tables): print(tables[["name","schema_name"]].to_string(index=False))

## Step 3 — Apply Snowflake Horizon Governance

Masking policy on `credit_card` — independent of any Databricks UC policies.

**Idempotent pattern:** UNSET before SET handles both first run and re-runs without errors.

In [ ]:
sf_session.sql(f"CREATE SCHEMA IF NOT EXISTS {SF_GOVERNANCE_DB}.GOVERNANCE_POLICIES").collect()

sf_session.sql(f"""
    CREATE MASKING POLICY IF NOT EXISTS {SF_GOVERNANCE_DB}.GOVERNANCE_POLICIES.MASK_CREDIT_CARD
        AS (val STRING) RETURNS STRING ->
        CASE
            WHEN CURRENT_ROLE() = 'ACCOUNTADMIN' THEN val
            ELSE CONCAT('****-****-****-', RIGHT(val, 4))
        END
""").collect()

# Idempotent apply: UNSET first (catches re-runs), then SET
try:
    sf_session.sql(f"""
        ALTER ICEBERG TABLE {TBL_SENSITIVE}
        MODIFY COLUMN "credit_card" UNSET MASKING POLICY
    """).collect()
except Exception:
    pass  # Not yet applied on first run — safe to ignore

sf_session.sql(f"""
    ALTER ICEBERG TABLE {TBL_SENSITIVE}
    MODIFY COLUMN "credit_card"
    SET MASKING POLICY {SF_GOVERNANCE_DB}.GOVERNANCE_POLICIES.MASK_CREDIT_CARD
""").collect()
print("Masking policy MASK_CREDIT_CARD applied to credit_card ✅")

## Step 4 — Grants

In [ ]:
sf_session.sql(f"CREATE ROLE IF NOT EXISTS {SF_READER_ROLE}").collect()
sf_session.sql(f"GRANT USAGE  ON DATABASE {SF_FEDERATED_DB} TO ROLE {SF_READER_ROLE}").collect()
sf_session.sql(f"GRANT USAGE  ON SCHEMA   {SF_FEDERATED_DB}.{SF_FEDERATED_SCHEMA} TO ROLE {SF_READER_ROLE}").collect()
sf_session.sql(f"GRANT SELECT ON ALL ICEBERG TABLES IN SCHEMA {SF_FEDERATED_DB}.{SF_FEDERATED_SCHEMA} TO ROLE {SF_READER_ROLE}").collect()
sf_session.sql(f"GRANT ROLE {SF_READER_ROLE} TO USER {SF_USERNAME}").collect()
print(f"Grants applied for {SF_READER_ROLE} ✅")

---
## Demo 1 — CUSTOMER_ORDERS: Open Table `[DataFrame]`
No governance policy — all roles see identical data.

In [ ]:
switch_role("ACCOUNTADMIN")
df_orders = spark.sql(f"SELECT * FROM {TBL_ORDERS}")
print(f"[DataFrame] CUSTOMER_ORDERS — {df_orders.count()} rows")
df_orders.orderBy("order_id").show(truncate=False)

## Demo 2 — Filter + Select `[DataFrame]`

In [ ]:
print("[DataFrame] .filter().select() — SHIPPED / DELIVERED only:")
(
    df_orders
    .filter(col("status").isin("SHIPPED", "DELIVERED"))
    .select("order_id", "product", "amount", "status")
    .orderBy("amount", ascending=False)
    .show(truncate=False)
)

## Demo 3 — Aggregation `[DataFrame]`

In [ ]:
print("[DataFrame] .groupBy().agg() — revenue by status:")
(
    df_orders
    .groupBy("status")
    .agg(count("*").alias("order_count"), spark_sum("amount").alias("total_revenue"))
    .orderBy("total_revenue", ascending=False)
    .show(truncate=False)
)

---
## Demo 4 — SENSITIVE_ORDERS: ACCOUNTADMIN (Unmasked) `[DataFrame]`
`CURRENT_ROLE() = 'ACCOUNTADMIN'` → real credit card number.

In [ ]:
switch_role("ACCOUNTADMIN")
print("[DataFrame] SENSITIVE_ORDERS — ACCOUNTADMIN (credit_card unmasked):")
(
    spark.sql(f"SELECT * FROM {TBL_SENSITIVE}")
    .select("order_id", "customer_name", "credit_card", "amount", "region")
    .orderBy("order_id")
    .show(truncate=False)
)

## Demo 5 — SENSITIVE_ORDERS: Reader Role (Masked) `[DataFrame]`
Masking fires: `credit_card` → `****-****-****-XXXX`

In [ ]:
switch_role(SF_READER_ROLE)
df_sensitive = spark.sql(f"SELECT * FROM {TBL_SENSITIVE}")
print(f"[DataFrame] SENSITIVE_ORDERS — {SF_READER_ROLE} (masked):")
(
    df_sensitive
    .select("order_id", "customer_name", "credit_card", "amount", "region")
    .orderBy("order_id")
    .show(truncate=False)
)

## Demo 6 — Side-by-Side Governance `[DataFrame]`
Same Parquet files — different `credit_card` per role.

In [ ]:
COLS = ["order_id", "customer_name", "credit_card"]

switch_role("ACCOUNTADMIN")
admin_rows = spark.sql(f"SELECT * FROM {TBL_SENSITIVE}").select(*COLS).collect()

switch_role(SF_READER_ROLE)
reader_rows = spark.sql(f"SELECT * FROM {TBL_SENSITIVE}").select(*COLS).collect()

admin_pd  = pd.DataFrame([r.asDict() for r in admin_rows]); admin_pd.insert(0, "role", "ACCOUNTADMIN")
reader_pd = pd.DataFrame([r.asDict() for r in reader_rows]); reader_pd.insert(0, "role", SF_READER_ROLE)

combined = pd.concat([admin_pd, reader_pd]).sort_values(["order_id","role"]).reset_index(drop=True)
print("[Governance] Same Parquet files — Horizon masking enforced by Snowflake SQL engine:")
print(combined.to_string(index=False))

## Demo 7 — Join: ACCOUNTADMIN `[DataFrame]`
`credit_card` unmasked in join result.

In [ ]:
switch_role("ACCOUNTADMIN")
df_o = spark.sql(f"SELECT * FROM {TBL_ORDERS}")
df_s = spark.sql(f"SELECT * FROM {TBL_SENSITIVE}")
print("[DataFrame .join()] ACCOUNTADMIN — credit_card unmasked:")
(
    df_o.alias("o").join(df_s.alias("s"), col("o.customer_id") == col("s.customer_id"), "inner")
    .select(col("o.order_id"), col("o.product"), col("o.amount").alias("product_amount"),
            col("s.customer_name"), col("s.credit_card"), col("s.region"))
    .orderBy("order_id").show(truncate=False)
)

## Demo 8 — Join: Reader Role `[DataFrame]`
Masking enforced inside the join.

In [ ]:
switch_role(SF_READER_ROLE)
df_o = spark.sql(f"SELECT * FROM {TBL_ORDERS}")
df_s = spark.sql(f"SELECT * FROM {TBL_SENSITIVE}")
print(f"[DataFrame .join()] {SF_READER_ROLE} — credit_card masked:")
(
    df_o.alias("o").join(df_s.alias("s"), col("o.customer_id") == col("s.customer_id"), "inner")
    .select(col("o.order_id"), col("o.product"), col("o.amount").alias("product_amount"),
            col("s.customer_name"), col("s.credit_card"), col("s.region"))
    .orderBy("order_id").show(truncate=False)
)

## Demo 9 — Sync Catalog-Linked Database
`RESUME DISCOVERY` forces an immediate re-poll of the remote catalog.

In [ ]:
switch_role("ACCOUNTADMIN")
sf_session.sql(f"ALTER DATABASE {SF_FEDERATED_DB} RESUME DISCOVERY").collect()
status = sf_session.sql(f"SELECT SYSTEM$CATALOG_LINK_STATUS('{SF_FEDERATED_DB}')").collect()[0][0]
print(f"Sync status: {status}")

---
## Summary

| Demo | API | Role | Result |
|------|-----|------|--------|
| 1 — Read orders | `spark.sql()` | ACCOUNTADMIN | All rows |
| 2 — Filter | `df.filter().select()` | ACCOUNTADMIN | SHIPPED/DELIVERED |
| 3 — Aggregate | `df.groupBy().agg()` | ACCOUNTADMIN | Revenue by status |
| 4 — Read sensitive | `spark.sql()` | ACCOUNTADMIN | `4111-1111-1111-1111` |
| 5 — Read sensitive | `spark.sql()` | {SF_READER_ROLE} | `****-****-****-1111` |
| 6 — Side-by-side | collect + pandas | Both | Governance comparison |
| 7 — Join | `df.join()` | ACCOUNTADMIN | credit_card unmasked |
| 8 — Join | `df.join()` | {SF_READER_ROLE} | credit_card masked |
| 9 — Sync | `sf_session.sql()` | ACCOUNTADMIN | CLD re-polled |

**Snowflake Horizon masking is enforced by the SQL engine — applies to `.filter()`, `.select()`, `.join()` equally.**